# 02 — Preprocessing & Splitting

**Objective.** Create a stratified train/validation split from `train.csv` and define
preprocessing that is fitted on **training data only**. The split and a demonstration
of the fitted `ColumnTransformer` are saved for the next notebooks.

No preprocessing is fitted on `test.csv`: it is unlabelled and reserved for the final
submission.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()), Path.cwd())
RAW = ROOT / "data" / "raw" / "binaryclassificationwithabankchurndataset"
PROC = ROOT / "data" / "processed"
MODELS = ROOT / "models"
FIG = MODELS / "figures"
for d in (PROC, FIG):
    d.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

project root: /home/sabina/portfolio/creditscoring


In [2]:
DROP = ["id", "CustomerId", "Surname"]
df = pd.read_csv(RAW / "train.csv").drop(columns=DROP)
X = df.drop(columns=["Exited"])
y = df["Exited"]
print("Features:", X.columns.tolist())
print("Class balance:", y.value_counts(normalize=True).round(4).to_dict())

Features: ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Class balance: {0.0: 0.8015, 1.0: 0.1985}


In [3]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
train = X_train.copy(); train["Exited"] = y_train
val = X_val.copy();   val["Exited"] = y_val
train.to_csv(PROC / "train.csv", index=False)
val.to_csv(PROC / "val.csv", index=False)
print(f"Train: {X_train.shape[0]:,} rows | churn {y_train.mean():.1%}")
print(f"Val  : {X_val.shape[0]:,} rows | churn {y_val.mean():.1%}")

Train: 12,000 rows | churn 19.9%
Val  : 3,000 rows | churn 19.8%


## Preprocessor (demonstration, fitted on train only)

Numeric → median imputation + standardisation. Categorical → most-frequent
imputation + one-hot encoding with `handle_unknown='ignore'` so unseen validation /
inference categories never crash scoring. Notebook 03 rebuilds the same transformer
inside each candidate model's `Pipeline`, guaranteeing identical transforms.

In [4]:
num = X.select_dtypes(include=np.number).columns.tolist()
cat = X.select_dtypes(exclude=np.number).columns.tolist()
print("Numeric:", num)
print("Categorical:", cat)

def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), num),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat),
    ])

prep = make_preprocessor()
Xt = prep.fit_transform(X_train)
print("Transformed train matrix:", Xt.shape)
print("Zero-variance check — all finite:", np.isfinite(Xt).all())

Numeric: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Categorical: ['Geography', 'Gender']


Transformed train matrix: (12000, 13)
Zero-variance check — all finite: True


**Outcome.** A reproducible 80/20 stratified split is saved to `data/processed/`,
and the preprocessing contract is fixed (train-only fitting, unknown-safe encoding).
**Next:** 03_training — benchmark candidates under cross-validation.